#Biliotheken, Funktionen

In [ ]:
import numpy as np
import pandas as pd
import datetime
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
def descr_stat(df, cols=None, exclude=None):
    exclude = exclude or []
    if cols is None:
        cols = df.select_dtypes(include=['object']).columns.tolist()
    else:
        cols = [c for c in cols if c in df.columns]
    cols = [c for c in cols if c not in exclude]

    for c in cols:
        vc_df = df[c].value_counts().rename_axis('value').reset_index(name='count')
        total = int(vc_df['count'].sum()) if not vc_df.empty else 0
        vc_df['percent'] = (vc_df['count'] / total * 100).round(2) if total else 0.0
        print(f"\n\n=== {c} (unique: {vc_df.shape[0]}, total: {total}) ===")
        print(vc_df.to_string(index=False) if not vc_df.empty else " (нет значений)")

#1.1 Calls
##Datenbereinigung und -vorbereitung:
1. Entfernen Sie Duplikate und nicht relevante Spalten.
2. Behandeln Sie fehlende Werte angemessen.
3. Konvertieren Sie Datentypen (z.B. Datumsangaben und numerische Felder) in
geeignete Formate.

## Erste Datenübersicht



In [ ]:
df_calls = pd.read_excel('Calls (Done).xlsx', dtype={'CONTACTID': str, 'Id': str})
display(df_calls)

,Id,Call Start Time,Call Owner Name,CONTACTID,Call Type,Call Duration (in seconds),Call Status,Dialled Number,Outgoing Call Status,Scheduled in CRM,Tag
0,5805028000000805001,30.06.2023 08:43,John Doe,NaN,Inbound,171.0,Received,NaN,NaN,NaN,NaN
1,5805028000000768006,30.06.2023 08:46,John Doe,NaN,Outbound,28.0,Attended Dialled,NaN,Completed,0.0,NaN
2,5805028000000764027,30.06.2023 08:59,John Doe,NaN,Outbound,24.0,Attended Dialled,NaN,Completed,0.0,NaN
3,5805028000000787003,30.06.2023 09:20,John Doe,5805028000000645014,Outbound,6.0,Attended Dialled,NaN,Completed,0.0,NaN
4,5805028000000768019,30.06.2023 09:30,John Doe,5805028000000645014,Outbound,11.0,Attended Dialled,NaN,Completed,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...
95869,5805028000056889515,21.06.2024 15:30,Ulysses Adams,5805028000056564231,Outbound,6.0,Attended Dialled,NaN,Completed,0.0,NaN
95870,5805028000056875317,21.06.2024 15:30,Victor Barnes,5805028000054867023,Outbound,8.0,Attended Dialled,NaN,Completed,0.0,NaN
95871,5805028000056832495,21.06.2024 15:30,Kevin Parker,5805028000010617278,Outbound,5.0,Attended Dialled,NaN,Completed,0.0,NaN
95872,5805028000056893619,21.06.2024 15:30,Victor Barnes,5805028000056839048,Outbound,0.0,Unattended Dialled,NaN,Completed,0.0,NaN


In [ ]:
missing_percent = df_calls.isnull().mean() * 100
missing_percent = missing_percent.round(1)

subset = df_calls.isnull().head(300).astype(int)

fig = go.Figure(data=go.Heatmap(
    z=subset.values,
    x=[f"{col}<br>({missing_percent[col]}%)" for col in df_calls.columns],
    colorscale=[[0, 'white'], [1, 'black']],
    showscale=False
))

fig.update_layout(
    title="Hitmap of missing values",
    height=500,
)

fig.show()

Entfernung leerer Spalten 'Dialled Number' и 'Tag'





In [ ]:
df_calls.drop(['Dialled Number', 'Tag'], axis=1, inplace=True)

##Duplikatbereinigung

In [ ]:
df_calls[df_calls.duplicated(subset=df_calls.columns[1:])].shape

(3257, 9)

In [ ]:
df_calls.drop_duplicates(subset=df_calls.columns[1:], inplace=True)

In [ ]:
df_calls[df_calls.duplicated(subset=df_calls.columns[1:])].shape

(0, 9)

##Behandlung fehlende Werte

In [ ]:
display(df_calls.isnull().sum())

,0
Id,0
Call Start Time,0
Call Owner Name,0
CONTACTID,3802
Call Type,0
Call Duration (in seconds),79
Call Status,0
Outgoing Call Status,8813
Scheduled in CRM,8813


In [ ]:
null_dur = df_calls[df_calls['Call Duration (in seconds)'].isnull()]
null_dur.info()

<class 'pandas.core.frame.DataFrame'>
Index: 79 entries, 21711 to 95868
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Id                          79 non-null     object 
 1   Call Start Time             79 non-null     object 
 2   Call Owner Name             79 non-null     object 
 3   CONTACTID                   5 non-null      object 
 4   Call Type                   79 non-null     object 
 5   Call Duration (in seconds)  0 non-null      float64
 6   Call Status                 79 non-null     object 
 7   Outgoing Call Status        79 non-null     object 
 8   Scheduled in CRM            79 non-null     float64
dtypes: float64(2), object(7)
memory usage: 6.2+ KB


In [ ]:
null_dur

,Id,Call Start Time,Call Owner Name,CONTACTID,Call Type,Call Duration (in seconds),Call Status,Outgoing Call Status,Scheduled in CRM
21711,5805028000012419809,15.11.2023 15:00,Kevin Parker,NaN,Outbound,NaN,Cancelled,Cancelled,1.0
36095,5805028000012419971,15.01.2024 15:00,Kevin Parker,NaN,Outbound,NaN,Overdue,Overdue,1.0
37600,5805028000026957356,19.01.2024 10:00,Victor Barnes,NaN,Outbound,NaN,Overdue,Overdue,1.0
37607,5805028000026466529,19.01.2024 10:30,Victor Barnes,5805028000021065227,Outbound,NaN,Overdue,Overdue,1.0
37623,5805028000026957294,19.01.2024 11:00,Victor Barnes,NaN,Outbound,NaN,Overdue,Overdue,1.0
...,...,...,...,...,...,...,...,...,...
89073,5805028000053636072,05.06.2024 17:00,Victor Barnes,NaN,Outbound,NaN,Overdue,Overdue,1.0
89257,5805028000053651109,06.06.2024 07:00,Victor Barnes,NaN,Outbound,NaN,Cancelled,Cancelled,1.0
92959,5805028000055092068,14.06.2024 16:00,Victor Barnes,NaN,Outbound,NaN,Scheduled,Scheduled,1.0
93676,5805028000055626456,17.06.2024 10:00,Victor Barnes,NaN,Outbound,NaN,Scheduled,Scheduled,1.0


In [ ]:
null_dur[df_calls['CONTACTID'].notnull()]

/tmp/ipython-input-1361229812.py:1: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.



,Id,Call Start Time,Call Owner Name,CONTACTID,Call Type,Call Duration (in seconds),Call Status,Outgoing Call Status,Scheduled in CRM
37607,5805028000026466529,19.01.2024 10:30,Victor Barnes,5805028000021065227,Outbound,NaN,Overdue,Overdue,1.0
38383,5805028000027542172,22.01.2024 10:00,Victor Barnes,5805028000026965161,Outbound,NaN,Overdue,Overdue,1.0
71821,5805028000045571004,22.04.2024 18:00,Victor Barnes,5805028000030143280,Outbound,NaN,Overdue,Overdue,1.0
73139,5805028000046290006,25.04.2024 12:00,Quincy Vincent,5805028000018056036,Outbound,NaN,Overdue,Overdue,1.0
80048,5805028000049918362,14.05.2024 14:30,Victor Barnes,5805028000012498013,Outbound,NaN,Overdue,Overdue,1.0


Bei der Untersuchung von 79 Ausfällen in der Anrufdauer (in Sekunden) lässt sich sagen, dass es sich bei 5 davon um Versuche handelt, bestimmte Personen zu erreichen, wobei der Anruf abgebrochen wurde. Die übrigen 74 sind Anrufe, die aus irgendeinem Grund nicht zustande gekommen sind. Ein Ersatz durch den Mittelwert oder Median bringt in diesem Fall nichts.

In [ ]:
null_contactid = df_calls[df_calls['CONTACTID'].isnull()]
null_contactid.describe()

,Call Duration (in seconds),Scheduled in CRM
count,3728.000000,2658.000000
mean,153.426770,0.048909
std,370.524565,0.215718
min,0.000000,0.000000
25%,0.000000,0.000000
50%,6.000000,0.000000
75%,91.000000,0.000000
max,3386.000000,1.000000


In [ ]:
null_contactid

,Id,Call Start Time,Call Owner Name,CONTACTID,Call Type,Call Duration (in seconds),Call Status,Outgoing Call Status,Scheduled in CRM
0,5805028000000805001,30.06.2023 08:43,John Doe,NaN,Inbound,171.0,Received,NaN,NaN
1,5805028000000768006,30.06.2023 08:46,John Doe,NaN,Outbound,28.0,Attended Dialled,Completed,0.0
2,5805028000000764027,30.06.2023 08:59,John Doe,NaN,Outbound,24.0,Attended Dialled,Completed,0.0
7,5805028000000879006,03.07.2023 13:06,Jane Smith,NaN,Outbound,0.0,Unattended Dialled,Completed,0.0
8,5805028000000870005,03.07.2023 13:08,Jane Smith,NaN,Outbound,40.0,Attended Dialled,Completed,0.0
...,...,...,...,...,...,...,...,...,...
95852,5805028000056849471,21.06.2024 15:06,Ulysses Adams,NaN,Outbound,7.0,Attended Dialled,Completed,0.0
95853,5805028000056859477,21.06.2024 15:07,Ulysses Adams,NaN,Inbound,407.0,Received,NaN,NaN
95861,5805028000056876318,21.06.2024 15:20,Eva Kent,NaN,Outbound,5.0,Attended Dialled,Completed,0.0
95868,5805028000056912329,21.06.2024 15:30,Victor Barnes,NaN,Outbound,NaN,Scheduled,Scheduled,1.0


Es wurden 3800 Anrufe ohne Contact ID gefunden.
Das kann bedeuten, dass:

die Anrufe von Nummern kamen, die nicht im CRM registriert sind,

es zu Fehlern beim Speichern der Kontaktdaten kam,

es sich um interne oder erfolglose Anrufe handelte.
Zur Klärung sollte man die Häufigkeit solcher Einträge nach Managern, Anrufstatus und Zeit vergleichen, um mögliche systemische oder prozessbezogene Ursachen zu ermitteln.

Übersetzt mit DeepL.com (kostenlose Version)

In [ ]:
missing_ = df_calls.isnull().mean() * 100
missing_ = missing_.round(1)

subset = df_calls.isnull().head(500).astype(int)

fig = go.Figure(data=go.Heatmap(
    z=subset.values,
    x=[f"{col}<br>({missing_percent[col]}%)" for col in df_calls.columns],  # подписи с %
    colorscale=[[0, 'white'], [1, 'black']],
    showscale=False
))

fig.update_layout(
    title="Hitmap of missing values",
    height=500,
)

fig.show()

## Konvertierung von Datentypen

In [ ]:
df_calls['Call Start Time'] = df_calls['Call Start Time'].apply(pd.to_datetime, errors='coerce')

/tmp/ipython-input-589503052.py:1: UserWarning:

Parsing dates in %d.%m.%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.



In [ ]:
df_calls.info()

<class 'pandas.core.frame.DataFrame'>
Index: 92617 entries, 0 to 95873
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   Id                          92617 non-null  object        
 1   Call Start Time             92617 non-null  datetime64[ns]
 2   Call Owner Name             92617 non-null  object        
 3   CONTACTID                   88815 non-null  object        
 4   Call Type                   92617 non-null  object        
 5   Call Duration (in seconds)  92538 non-null  float64       
 6   Call Status                 92617 non-null  object        
 7   Outgoing Call Status        83804 non-null  object        
 8   Scheduled in CRM            83804 non-null  float64       
dtypes: datetime64[ns](1), float64(2), object(6)
memory usage: 9.1+ MB


#1.2 Calls
##Deskriptive Statistik:
1. Berechnen Sie zusammenfassende Statistiken (Mittelwert, Median, Modus,
Spannweite) für numerische Felder.
2. Analysieren Sie kategoriale Felder wie Qualität, Phase, Quelle und Produkt.

## Deskriptive Statistik für Numerische Feleder

In [ ]:
numb_cols = df_calls.select_dtypes(include=['int64', 'float64']).columns
numb_cols

Index(['Call Duration (in seconds)', 'Scheduled in CRM'], dtype='object')

In [ ]:
df_calls.describe()

,Call Start Time,Call Duration (in seconds),Scheduled in CRM
count,92617,92538.000000,83804.000000
mean,2024-02-15 06:15:02.480537856,170.330178,0.001623
min,2023-01-08 09:39:00,0.000000,0.000000
25%,2023-11-17 15:22:00,4.000000,0.000000
50%,2024-02-27 15:29:00,9.000000,0.000000
75%,2024-05-17 10:25:00,107.000000,0.000000
max,2024-12-06 21:18:00,7625.000000,1.000000
std,NaN,406.944468,0.040252


In [ ]:
fig = px.strip(
    df_calls, x='Call Owner Name',
    y='Call Duration (in seconds)'
)
fig.show()

##Deskriptive Statistik für Kategorische Feleder

In [ ]:
categ_calls = df_calls.select_dtypes(include=['object']).columns
categ_calls

Index(['Id', 'Call Owner Name', 'CONTACTID', 'Call Type', 'Call Status',
       'Outgoing Call Status'],
      dtype='object')

In [ ]:
descr_stat(df_calls, cols=categ_calls, exclude=['Id', 'CONTACTID'])



=== Call Owner Name (unique: 33, total: 92617) ===
          value  count  percent
   Yara Edwards   8532     9.21
   Julia Nelson   7213     7.79
     Ian Miller   7027     7.59
  Charlie Davis   6943     7.50
    Diana Evans   6713     7.25
  Ulysses Adams   5961     6.44
      Amy Green   5575     6.02
  Victor Barnes   5361     5.79
   Kevin Parker   5359     5.79
     Nina Scott   5317     5.74
Paula Underwood   4498     4.86
 Quincy Vincent   4223     4.56
     Jane Smith   3537     3.82
   Cara Iverson   3019     3.26
       John Doe   2956     3.19
       Ben Hall   2850     3.08
  Alice Johnson   1244     1.34
  Mason Roberts   1155     1.25
    Derek James    941     1.02
    George King    828     0.89
 Zachary Foster    518     0.56
       Eva Kent    492     0.53
      Sam Young    456     0.49
  Fiona Jackson    451     0.49
   Rachel White    425     0.46
    Xander Dean    294     0.32
   Ethan Harris    277     0.30
     Hannah Lee    175     0.19
    Wendy Clark    

In [ ]:
df_calls

,Id,Call Start Time,Call Owner Name,CONTACTID,Call Type,Call Duration (in seconds),Call Status,Outgoing Call Status,Scheduled in CRM
0,5805028000000805001,2023-06-30 08:43:00,John Doe,NaN,Inbound,171.0,Received,NaN,NaN
1,5805028000000768006,2023-06-30 08:46:00,John Doe,NaN,Outbound,28.0,Attended Dialled,Completed,0.0
2,5805028000000764027,2023-06-30 08:59:00,John Doe,NaN,Outbound,24.0,Attended Dialled,Completed,0.0
3,5805028000000787003,2023-06-30 09:20:00,John Doe,5805028000000645014,Outbound,6.0,Attended Dialled,Completed,0.0
4,5805028000000768019,2023-06-30 09:30:00,John Doe,5805028000000645014,Outbound,11.0,Attended Dialled,Completed,0.0
...,...,...,...,...,...,...,...,...,...
95869,5805028000056889515,2024-06-21 15:30:00,Ulysses Adams,5805028000056564231,Outbound,6.0,Attended Dialled,Completed,0.0
95870,5805028000056875317,2024-06-21 15:30:00,Victor Barnes,5805028000054867023,Outbound,8.0,Attended Dialled,Completed,0.0
95871,5805028000056832495,2024-06-21 15:30:00,Kevin Parker,5805028000010617278,Outbound,5.0,Attended Dialled,Completed,0.0
95872,5805028000056893619,2024-06-21 15:30:00,Victor Barnes,5805028000056839048,Outbound,0.0,Unattended Dialled,Completed,0.0


##Datenspeicherung

In [ ]:
df_calls.to_pickle('Cl_Calls.pkl')

#2.1 Contacts
##Datenbereinigung und -vorbereitung:
1. Entfernen Sie Duplikate und nicht relevante Spalten.
2. Behandeln Sie fehlende Werte angemessen.
3. Konvertieren Sie Datentypen (z.B. Datumsangaben und numerische Felder) in
geeignete Formate.

## Erste Datenübersicht

In [ ]:
df_cont = pd.read_excel('Contacts (Done).xlsx', dtype={'Id': str})
display(df_cont)

,Id,Contact Owner Name,Created Time,Modified Time
0,5805028000000645014,Rachel White,27.06.2023 11:28,22.12.2023 13:34
1,5805028000000872003,Charlie Davis,03.07.2023 11:31,21.05.2024 10:23
2,5805028000000889001,Bob Brown,02.07.2023 22:37,21.12.2023 13:17
3,5805028000000907006,Bob Brown,03.07.2023 05:44,29.12.2023 15:20
4,5805028000000939010,Nina Scott,04.07.2023 10:11,16.04.2024 16:14
...,...,...,...,...
18543,5805028000056889209,Ulysses Adams,21.06.2024 12:11,21.06.2024 14:11
18544,5805028000056889351,Eva Kent,21.06.2024 13:32,21.06.2024 15:32
18545,5805028000056892018,Eva Kent,21.06.2024 10:21,21.06.2024 12:21
18546,5805028000056892055,Yara Edwards,21.06.2024 10:22,21.06.2024 12:23


In [ ]:
df_cont.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18548 entries, 0 to 18547
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Id                  18548 non-null  object
 1   Contact Owner Name  18548 non-null  object
 2   Created Time        18548 non-null  object
 3   Modified Time       18548 non-null  object
dtypes: object(4)
memory usage: 579.8+ KB


##Duplikatbereinigung

In [ ]:
df_cont[df_cont.duplicated(subset=df_cont.columns[1:])].shape

(38, 4)

In [ ]:
df_cont.drop_duplicates(subset=df_cont.columns[1:], inplace=True)

In [ ]:
df_cont[df_cont.duplicated(subset=df_cont.columns[1:])].shape

(0, 4)

##Behandlung fehlende Werte

In [ ]:
display(df_cont.isnull().sum())

,0
Id,0
Contact Owner Name,0
Created Time,0
Modified Time,0


## Konvertierung von Datentypen

In [ ]:
df_cont[['Created Time', 'Modified Time']] = df_cont[['Created Time', 'Modified Time']].apply(pd.to_datetime, errors='coerce')

/tmp/ipython-input-3957899212.py:1: UserWarning:

Parsing dates in %d.%m.%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.

/tmp/ipython-input-3957899212.py:1: UserWarning:

Parsing dates in %d.%m.%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.



In [ ]:
df_cont.info()

<class 'pandas.core.frame.DataFrame'>
Index: 18510 entries, 0 to 18547
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Id                  18510 non-null  object        
 1   Contact Owner Name  18510 non-null  object        
 2   Created Time        18510 non-null  datetime64[ns]
 3   Modified Time       18510 non-null  datetime64[ns]
dtypes: datetime64[ns](2), object(2)
memory usage: 723.0+ KB


#2.2 Contacts
##Deskriptive Statistik:
1. Berechnen Sie zusammenfassende Statistiken (Mittelwert, Median, Modus,
Spannweite) für numerische Felder.
2. Analysieren Sie kategoriale Felder wie Qualität, Phase, Quelle und Produkt.

##Deskriptive Statistik für Numerische Feleder

In [ ]:
df_cont.describe()

,Created Time,Modified Time
count,18510,18510
mean,2024-01-24 14:25:56.155591680,2024-02-15 09:11:05.273905920
min,2023-06-27 11:28:00,2023-07-06 10:54:00
25%,2023-11-15 16:49:15,2023-12-09 14:51:45
50%,2024-02-01 18:44:30,2024-02-29 01:12:30
75%,2024-04-12 16:15:45,2024-04-26 22:40:45
max,2024-06-21 15:30:00,2024-06-21 15:32:00


##Deskriptive Statistik für Kategorische Feleder

In [ ]:
categ_cols = df_cont.select_dtypes(include=['object']).columns
categ_cols

Index(['Id', 'Contact Owner Name'], dtype='object')

In [ ]:
descr_stat(df_cont, cols=categ_cols, exclude=['Id'])



=== Contact Owner Name (unique: 28, total: 18510) ===
          value  count  percent
  Charlie Davis   2018    10.90
  Ulysses Adams   1809     9.77
   Julia Nelson   1769     9.56
Paula Underwood   1486     8.03
 Quincy Vincent   1415     7.64
     Nina Scott   1148     6.20
       Ben Hall   1037     5.60
  Victor Barnes    966     5.22
   Cara Iverson    880     4.75
   Rachel White    768     4.15
     Jane Smith    754     4.07
     Ian Miller    682     3.68
      Bob Brown    679     3.67
    Diana Evans    677     3.66
   Yara Edwards    655     3.54
      Amy Green    620     3.35
       Eva Kent    365     1.97
   Kevin Parker    324     1.75
  Mason Roberts    217     1.17
    George King    144     0.78
      Sam Young     37     0.20
  Alice Johnson     27     0.15
  Oliver Taylor     19     0.10
 Zachary Foster      8     0.04
     Tina Zhang      2     0.01
    Wendy Clark      2     0.01
          False      1     0.01
    Derek James      1     0.01


##Datenspeicherung

In [ ]:
df_cont.to_pickle('Cl_Cont.pkl')

#3.1 Spend
##Datenbereinigung und -vorbereitung:
1. Entfernen Sie Duplikate und nicht relevante Spalten.
2. Behandeln Sie fehlende Werte angemessen.
3. Konvertieren Sie Datentypen (z.B. Datumsangaben und numerische Felder) in
geeignete Formate.

## Erste Datenübersicht

In [ ]:
df_spend = pd.read_excel('Spend (Done).xlsx')
display(df_spend)

,Date,Source,Campaign,Impressions,Spend,Clicks,AdGroup,Ad
0,2023-07-03,Google Ads,gen_analyst_DE,6,0.00,0,NaN,NaN
1,2023-07-03,Google Ads,performancemax_eng_DE,4,0.01,1,NaN,NaN
2,2023-07-03,Facebook Ads,NaN,0,0.00,0,NaN,NaN
3,2023-07-03,Google Ads,NaN,0,0.00,0,NaN,NaN
4,2023-07-03,CRM,NaN,0,0.00,0,NaN,NaN
...,...,...,...,...,...,...,...,...
20774,2024-06-21,Facebook Ads,17.03.24wide_AT,7,0.07,0,wide,bloggersvideo16com_at
20775,2024-06-21,Tiktok Ads,12.07.2023wide_DE,61,0.16,0,wide,bloggersvideo14com
20776,2024-06-21,Partnership,NaN,0,0.00,0,NaN,NaN
20777,2024-06-21,Tiktok Ads,NaN,0,0.00,0,NaN,NaN


In [ ]:
df_spend.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20779 entries, 0 to 20778
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   Date         20779 non-null  datetime64[ns]
 1   Source       20779 non-null  object        
 2   Campaign     14785 non-null  object        
 3   Impressions  20779 non-null  int64         
 4   Spend        20779 non-null  float64       
 5   Clicks       20779 non-null  int64         
 6   AdGroup      13951 non-null  object        
 7   Ad           13951 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(2), object(4)
memory usage: 1.3+ MB


In [ ]:
missing_percent = df_spend.isnull().mean() * 100
missing_percent = missing_percent.round(1)

subset = df_spend.isnull().head(500).astype(int)

fig = go.Figure(data=go.Heatmap(
    z=subset.values,
    x=[f"{col}<br>({missing_percent[col]}%)" for col in df_spend.columns],
    colorscale=[[0, 'white'], [1, 'black']],
    showscale=False
))

fig.update_layout(
    title="Hitmap of missing values",
    height=500,
)

fig.show()

##Duplikatbereinigung



In [ ]:
df_spend[df_spend.duplicated(subset=df_spend.columns)].shape

(917, 8)

Zuerst löschen wir die Spalten, in denen die Zahlenwerte gleich 0 sind, dann die Duplikate.

In [ ]:
df_spend[
    (df_spend['Impressions'] == 0) &
    (df_spend['Spend'] == 0) &
    (df_spend['Clicks'] == 0)
]

,Date,Source,Campaign,Impressions,Spend,Clicks,AdGroup,Ad
2,2023-07-03,Facebook Ads,NaN,0,0.0,0,NaN,NaN
3,2023-07-03,Google Ads,NaN,0,0.0,0,NaN,NaN
4,2023-07-03,CRM,NaN,0,0.0,0,NaN,NaN
7,2023-07-03,Bloggers,NaN,0,0.0,0,NaN,NaN
8,2023-07-03,Youtube Ads,NaN,0,0.0,0,NaN,NaN
...,...,...,...,...,...,...,...,...
20764,2024-06-21,Telegram posts,NaN,0,0.0,0,NaN,NaN
20770,2024-06-21,Organic,NaN,0,0.0,0,NaN,NaN
20773,2024-06-21,Organic,NaN,0,0.0,0,NaN,NaN
20776,2024-06-21,Partnership,NaN,0,0.0,0,NaN,NaN


In [ ]:
df_spend = df_spend[
    ~((df_spend['Impressions'] == 0) &
      (df_spend['Spend'] == 0) &
      (df_spend['Clicks'] == 0))
]

In [ ]:
df_spend[df_spend.duplicated(subset=df_spend)].shape

(7, 8)

In [ ]:
df_spend.drop_duplicates(subset=df_spend.columns, inplace=True)

In [ ]:
df_spend[df_spend.duplicated(subset=df_spend)].shape

(0, 8)

##Entfernung der Zeilen mit dem Wert Test

In [ ]:
df_spend[df_spend['Source'] == 'Test'].shape

(169, 8)

In [ ]:
df_spend = df_spend[df_spend['Source'] != 'Test']
df_spend[df_spend['Source'] == 'Test'].shape

(0, 8)

##Behandlung fehlende Werte

In [ ]:
df_spend.isnull().sum()

,0
Date,0
Source,0
Campaign,2196
Impressions,0
Spend,0
Clicks,0
AdGroup,3030
Ad,3030


In [ ]:
df_spend.fillna('Unknown', inplace=True)

In [ ]:
missing_percent = df_spend.isnull().mean() * 100
missing_percent = missing_percent.round(1)

subset = df_spend.isnull().head(200).astype(int)

fig = go.Figure(data=go.Heatmap(
    z=subset.values,
    x=[f"{col}<br>({missing_percent[col]}%)" for col in df_spend.columns],
    colorscale=[[0, 'white'], [1, 'black']],
    showscale=False
))

fig.update_layout(
    title="Hitmap of missing values",
    height=300,
)

fig.show()

## Konvertierung von Datentypen

In [ ]:
df_spend["Date"] = pd.to_datetime(df_spend["Date"], errors="raise")

In [ ]:
df_spend.info()

<class 'pandas.core.frame.DataFrame'>
Index: 16231 entries, 0 to 20778
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   Date         16231 non-null  datetime64[ns]
 1   Source       16231 non-null  object        
 2   Campaign     16231 non-null  object        
 3   Impressions  16231 non-null  int64         
 4   Spend        16231 non-null  float64       
 5   Clicks       16231 non-null  int64         
 6   AdGroup      16231 non-null  object        
 7   Ad           16231 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(2), object(4)
memory usage: 1.1+ MB


#3.2 Spend
##Deskriptive Statistik:
1. Berechnen Sie zusammenfassende Statistiken (Mittelwert, Median, Modus,
Spannweite) für numerische Felder.
2. Analysieren Sie kategoriale Felder wie Qualität, Phase, Quelle und Produkt.

##Deskriptive Statistik für Numerische Feleder

In [ ]:
df_spend.describe().round(2)

,Date,Impressions,Spend,Clicks
count,16231,16231.00,16231.00,16231.00
mean,2024-01-11 17:36:44.004682240,3144.29,9.17,30.63
min,2023-07-03 00:00:00,0.00,0.00,0.00
25%,2023-10-09 00:00:00,15.00,0.12,0.00
50%,2024-01-24 00:00:00,205.00,1.75,4.00
75%,2024-04-12 00:00:00,958.00,8.12,17.00
max,2024-06-21 00:00:00,431445.00,774.00,2415.00
std,NaN,12863.41,29.98,95.38


##Deskriptive Statistik für Kategorische Feleder

In [ ]:
categ_cols = df_spend.select_dtypes(include=['object']).columns
categ_cols

Index(['Source', 'Campaign', 'AdGroup', 'Ad'], dtype='object')

In [ ]:
descr_stat(df_spend, cols=categ_cols)



=== Source (unique: 13, total: 16231) ===
         value  count  percent
  Facebook Ads   9195    56.65
    Tiktok Ads   2200    13.55
   Youtube Ads   1429     8.80
    Google Ads    911     5.61
       Organic    505     3.11
           SMM    483     2.98
Telegram posts    481     2.96
           CRM    315     1.94
       Webinar    308     1.90
      Bloggers    277     1.71
   Partnership    105     0.65
       Offline     16     0.10
         Radio      6     0.04


=== Campaign (unique: 51, total: 16231) ===
                      value  count  percent
                    Unknown   2196    13.53
          12.07.2023wide_DE   1811    11.16
            02.07.23wide_DE   1680    10.35
   04.07.23recentlymoved_DE   1395     8.59
          youtube_shorts_DE   1223     7.53
             07.07.23LAL_DE   1180     7.27
              03.07.23women   1168     7.20
  12.09.23interests_Uxui_DE   1141     7.03
               15.07.23b_DE    528     3.25
     24.09.23retargeting_DE    502  

##Diagramme


In [ ]:
df_spend_grouped = df_spend.groupby('Source')['Spend'].count().reset_index()
fig = px.pie(
    df_spend_grouped,
    values='Spend',
    names='Source',
    title='Count Source',
    hole=.3
)

fig.update_layout(
    width=900,
    height=650,
    margin=dict(l=20, r=20, t=50, b=20)
)

fig.show()

In [ ]:
df_spend_grouped = df_spend.groupby('AdGroup')['Spend'].count().reset_index()
fig = px.pie(
    df_spend_grouped,
    values='Spend',
    names='AdGroup',
    title='Count AdGroup',
    hole=.3)

fig.update_layout(
    width=900,
    height=600,
    margin=dict(l=20, r=20, t=50, b=20)
)

fig.show()

##Datenspeicherung



In [ ]:
df_spend.to_pickle('Cl_Spend.pkl')

#4.1 Deals
##Datenbereinigung und -vorbereitung:
1. Entfernen Sie Duplikate und nicht relevante Spalten.
2. Behandeln Sie fehlende Werte angemessen.
3. Konvertieren Sie Datentypen (z.B. Datumsangaben und numerische Felder) in
geeignete Formate.

## Erste Datenübersicht

In [ ]:
df_deals = pd.read_excel(
    'Deals (Done).xlsx',
    dtype={'Id': str, 'Contact Name': str}
    )
df_deals

,Id,Deal Owner Name,Closing Date,Quality,Stage,Lost Reason,Page,Campaign,SLA,Content,...,Product,Education Type,Created Time,Course duration,Months of study,Initial Amount Paid,Offer Total Amount,Contact Name,City,Level of Deutsch
0,5805028000056864695,Ben Hall,NaN,NaN,New Lead,NaN,/eng/test,03.07.23women,NaN,v16,...,NaN,NaN,21.06.2024 15:30,NaN,NaN,NaN,NaN,5805028000056849495,NaN,NaN
1,5805028000056859489,Ulysses Adams,NaN,NaN,New Lead,NaN,/at-eng,NaN,NaN,NaN,...,Web Developer,Morning,21.06.2024 15:23,6.0,NaN,0,2000,5805028000056834471,NaN,NaN
2,5805028000056832357,Ulysses Adams,21.06.2024,D - Non Target,Lost,Non target,/at-eng,engwien_AT,00:26:43,b1-at,...,NaN,NaN,21.06.2024 14:45,NaN,NaN,NaN,NaN,5805028000056854421,NaN,NaN
3,5805028000056824246,Eva Kent,21.06.2024,E - Non Qualified,Lost,Invalid number,/eng,04.07.23recentlymoved_DE,01:00:04,bloggersvideo14com,...,NaN,NaN,21.06.2024 13:32,NaN,NaN,NaN,NaN,5805028000056889351,NaN,NaN
4,5805028000056873292,Ben Hall,21.06.2024,D - Non Target,Lost,Non target,/eng,discovery_DE,00:53:12,website,...,NaN,NaN,21.06.2024 13:21,NaN,NaN,NaN,NaN,5805028000056876176,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21590,5805028000000945016,Jane Smith,29.08.2023,A - High,Lost,Changed Decision,eng/digital-marketing,02.07.23wide_DE,"56 days, 19:01:59",b3,...,NaN,NaN,03.07.2023 20:39,NaN,NaN,NaN,NaN,5805028000000968001,NaN,NaN
21591,5805028000000927004,Bob Brown,09.07.2023,D - Non Target,Lost,Does not speak English,eng/digital-marketing,03.07.23women,NaN,b3,...,NaN,NaN,03.07.2023 20:17,NaN,NaN,NaN,NaN,5805028000000961001,NaN,NaN
21592,5805028000000922001,Bob Brown,03.07.2023,E - Non Qualified,Lost,Refugee,/,NaN,"4 days, 22:47:14",NaN,...,NaN,NaN,03.07.2023 17:03,NaN,NaN,0,0,5805028000001009140,NaN,NaN
21593,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
df_deals.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21595 entries, 0 to 21594
Data columns (total 23 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Id                   21593 non-null  object 
 1   Deal Owner Name      21564 non-null  object 
 2   Closing Date         14645 non-null  object 
 3   Quality              19340 non-null  object 
 4   Stage                21593 non-null  object 
 5   Lost Reason          16124 non-null  object 
 6   Page                 21593 non-null  object 
 7   Campaign             16067 non-null  object 
 8   SLA                  15533 non-null  object 
 9   Content              14147 non-null  object 
 10  Term                 12454 non-null  object 
 11  Source               21593 non-null  object 
 12  Payment Type         496 non-null    object 
 13  Product              3592 non-null   object 
 14  Education Type       3300 non-null   object 
 15  Created Time         21593 non-null 

In [ ]:
missing_percent = df_deals.isnull().mean() * 100
missing_percent = missing_percent.round(2)

subset = df_deals.isnull().head(300).astype(int)

fig = go.Figure(data=go.Heatmap(
    z=subset.values,
    x=[f"{col}<br>({missing_percent[col]}%)" for col in df_deals.columns],  # подписи с %
    colorscale=[[0, 'white'], [1, 'black']],
    showscale=False
))

fig.update_layout(
    title="Hitmap of missing values",
    height=500,
)

fig.show()

##Duplikatbereinigung

In [ ]:
df_deals[df_deals.duplicated(subset=df_deals.columns[1:])].shape

(9, 23)

In [ ]:
df_deals.drop_duplicates(subset=df_deals.columns[1:], inplace=True)

In [ ]:
df_deals[df_deals.duplicated(subset=df_deals.columns[1:])].shape

(0, 23)

Entfernung leerer



In [ ]:
df_deals.replace('#REF!', np.nan, inplace=True)

In [ ]:
df_deals.dropna(how='all', inplace=True)
df_deals.tail()

,Id,Deal Owner Name,Closing Date,Quality,Stage,Lost Reason,Page,Campaign,SLA,Content,...,Product,Education Type,Created Time,Course duration,Months of study,Initial Amount Paid,Offer Total Amount,Contact Name,City,Level of Deutsch
21588,5805028000000970006,Jane Smith,04.07.2023,E - Non Qualified,Lost,Duplicate,eng/digital-marketing,03.07.23women,NaN,b3,...,NaN,NaN,04.07.2023 07:10,NaN,NaN,NaN,NaN,5805028000000979006,NaN,NaN
21589,5805028000000948010,Jane Smith,29.08.2023,B - Medium,Lost,needs time to think,eng/digital-marketing,03.07.23women,NaN,b3,...,NaN,NaN,04.07.2023 07:10,NaN,NaN,NaN,NaN,5805028000000979006,NaN,NaN
21590,5805028000000945016,Jane Smith,29.08.2023,A - High,Lost,Changed Decision,eng/digital-marketing,02.07.23wide_DE,"56 days, 19:01:59",b3,...,NaN,NaN,03.07.2023 20:39,NaN,NaN,NaN,NaN,5805028000000968001,NaN,NaN
21591,5805028000000927004,Bob Brown,09.07.2023,D - Non Target,Lost,Does not speak English,eng/digital-marketing,03.07.23women,NaN,b3,...,NaN,NaN,03.07.2023 20:17,NaN,NaN,NaN,NaN,5805028000000961001,NaN,NaN
21592,5805028000000922001,Bob Brown,03.07.2023,E - Non Qualified,Lost,Refugee,/,NaN,"4 days, 22:47:14",NaN,...,NaN,NaN,03.07.2023 17:03,NaN,NaN,0,0,5805028000001009140,NaN,NaN


##Entfernung der Zeilen mit dem Wert Test, Duplicate

In [ ]:
df_deals[df_deals['Source'] == 'Test'].shape

(159, 23)

In [ ]:
df_deals = df_deals[df_deals['Source'] != 'Test']
df_deals[df_deals['Source'] == 'Test'].shape

(0, 23)

In [ ]:
df_deals[(df_deals['Lost Reason'] == 'Duplicate')].shape

(1764, 23)

In [ ]:
df_deals = df_deals[df_deals['Lost Reason'] != 'Duplicate']
df_deals[(df_deals['Lost Reason'] == 'Duplicate')].shape

(0, 23)

##Entfernung irrelevanter Daten

In [ ]:
df_deals['Product'].value_counts()

,count
Product,
Digital Marketing,1944
UX/UI Design,1004
Web Developer,565
Find yourself in IT,3
Data Analytics,1


In [ ]:
df_deals[df_deals['Product'] == 'Find yourself in IT']

,Id,Deal Owner Name,Closing Date,Quality,Stage,Lost Reason,Page,Campaign,SLA,Content,...,Product,Education Type,Created Time,Course duration,Months of study,Initial Amount Paid,Offer Total Amount,Contact Name,City,Level of Deutsch
6095,5805028000042448604,Julia Nelson,11.05.2024,D - Non Target,Payment Done,Expensive,/page,NaN,NaN,NaN,...,Find yourself in IT,NaN,08.04.2024 17:34,NaN,NaN,1,1,5805028000005448163,NaN,NaN
10067,5805028000032304003,Charlie Davis,09.05.2024,D - Non Target,Lost,Non target,/event,NaN,NaN,NaN,...,Find yourself in IT,NaN,15.02.2024 19:20,NaN,NaN,0,0,5805028000012657064,NaN,NaN
16302,5805028000017521117,Julia Nelson,08.05.2024,D - Non Target,Lost,Non target,/workshop,NaN,NaN,NaN,...,Find yourself in IT,NaN,17.11.2023 17:20,NaN,NaN,0,0,5805028000005448163,NaN,NaN


In [ ]:
df_deals = df_deals[df_deals['Product'] != 'Find yourself in IT']
df_deals[df_deals['Product'] == 'Find yourself in IT'].shape

(0, 23)

In [ ]:
df_deals[df_deals['Product'] == 'Data Analytics']

,Id,Deal Owner Name,Closing Date,Quality,Stage,Lost Reason,Page,Campaign,SLA,Content,...,Product,Education Type,Created Time,Course duration,Months of study,Initial Amount Paid,Offer Total Amount,Contact Name,City,Level of Deutsch
20914,5805028000003554209,Bob Brown,01.08.2023,D - Non Target,Lost,Went to Rivals,/,NaN,NaN,NaN,...,Data Analytics,NaN,01.08.2023 18:06,NaN,NaN,NaN,6000,5805028000003550126,NaN,NaN


In [ ]:
df_deals = df_deals[df_deals['Product'] != 'Data Analytics']
df_deals[df_deals['Product'] == 'Data Analytics'].shape

(0, 23)

## Konvertierung von Datentypen

In [ ]:
df_deals['Created Time'] = pd.to_datetime(df_deals['Created Time'], dayfirst=True, errors='coerce')
df_deals['Closing Date'] = pd.to_datetime(df_deals['Closing Date'], dayfirst=True, errors='coerce')

In [ ]:
mask = df_deals['Created Time'] > df_deals['Closing Date']
df_deals.loc[mask, ['Created Time', 'Closing Date']] = df_deals.loc[mask, ['Closing Date', 'Created Time']].values

df_deals[['Closing Date', 'Created Time', 'SLA']]

,Closing Date,Created Time,SLA
0,NaT,2024-06-21 15:30:00,NaN
1,NaT,2024-06-21 15:23:00,NaN
2,2024-06-21 14:45:00,2024-06-21 00:00:00,00:26:43
3,2024-06-21 13:32:00,2024-06-21 00:00:00,01:00:04
4,2024-06-21 13:21:00,2024-06-21 00:00:00,00:53:12
...,...,...,...
21587,2023-07-06 00:00:00,2023-07-04 09:34:00,NaN
21589,2023-08-29 00:00:00,2023-07-04 07:10:00,NaN
21590,2023-08-29 00:00:00,2023-07-03 20:39:00,"56 days, 19:01:59"
21591,2023-07-09 00:00:00,2023-07-03 20:17:00,NaN


In [ ]:
df_deals['SLA'].value_counts()

,count
SLA,
00:10:11,6
00:10:46,6
00:11:09,6
00:17:08,5
00:30:07,5
...,...
04:31:05,1
01:23:14,1
01:14:13,1


In [ ]:
def convert_to_seconds(x):
    if pd.isna(x):
        return np.nan
    elif isinstance(x, datetime.time):
        return x.hour * 3600 + x.minute * 60 + x.second
    elif isinstance(x, datetime.timedelta):
        return x.total_seconds()

In [ ]:
df_deals['SLA Seconds'] = df_deals["SLA"].apply(convert_to_seconds)
df_deals['SLA Seconds'].value_counts()

,count
SLA Seconds,
611.0,6
646.0,6
669.0,6
1028.0,5
1807.0,5
...,...
16265.0,1
4994.0,1
4453.0,1


In [ ]:
df_deals.drop(columns=['SLA'], inplace=True)

In [ ]:
vc1 = df_deals['Initial Amount Paid'].value_counts()
vc2 = df_deals['Offer Total Amount'].value_counts()

res = pd.concat([vc1, vc2], axis=1)
res.columns = ['Initial_count', 'Offer_count']

display(res)

,Initial_count,Offer_count
1000,2574.0,3.0
0,817.0,789.0
300,186.0,NaN
500,91.0,NaN
350,80.0,NaN
2000,57.0,62.0
11000,35.0,1822.0
200,30.0,NaN
11500,24.0,382.0
3500,20.0,129.0


In [ ]:
def clean_amount(value):
    if pd.isna(value):
        return None
    value = str(value)
    value = value.replace('€', '').strip()
    value = value.replace('.', '')
    if ',' in value:
        value = value.split(',')[0]
    try:
        return int(value)
    except ValueError:
        return None

In [ ]:
for col in df_deals[['Initial Amount Paid', 'Offer Total Amount']]:
    df_deals[col] = df_deals[col].apply(clean_amount)

In [ ]:
vc1 = df_deals['Initial Amount Paid'].value_counts()
vc2 = df_deals['Offer Total Amount'].value_counts()

res = pd.concat([vc1, vc2], axis=1)
res.columns = ['Initial_count', 'Offer_count']

display(res)

,Initial_count,Offer_count
1000.0,2574.0,3.0
0.0,817.0,789.0
300.0,186.0,NaN
500.0,91.0,NaN
350.0,80.0,NaN
2000.0,57.0,62.0
3500.0,36.0,129.0
11000.0,35.0,1822.0
200.0,30.0,NaN
11500.0,24.0,382.0


In [ ]:
cols = ['Initial Amount Paid', 'Offer Total Amount']
df_deals.loc[:, cols] = df_deals[cols].replace(0, np.nan)

In [ ]:
mask = df_deals['Initial Amount Paid'] > df_deals['Offer Total Amount']
df_deals.loc[mask, ['Initial Amount Paid', 'Offer Total Amount']] = df_deals.loc[mask, ['Offer Total Amount', 'Initial Amount Paid']].values

In [ ]:
df_deals.info()

<class 'pandas.core.frame.DataFrame'>
Index: 19657 entries, 0 to 21592
Data columns (total 23 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Id                   19657 non-null  object        
 1   Deal Owner Name      19628 non-null  object        
 2   Closing Date         13057 non-null  datetime64[ns]
 3   Quality              17426 non-null  object        
 4   Stage                19657 non-null  object        
 5   Lost Reason          14267 non-null  object        
 6   Page                 19657 non-null  object        
 7   Campaign             15462 non-null  object        
 8   Content              13640 non-null  object        
 9   Term                 11944 non-null  object        
 10  Source               19657 non-null  object        
 11  Payment Type         479 non-null    object        
 12  Product              3513 non-null   object        
 13  Education Type       3235 non-null  

##Behandlung fehlende Werte



In [ ]:
df_deals.isnull().sum()

,0
Id,0
Deal Owner Name,29
Closing Date,6600
Quality,2231
Stage,0
Lost Reason,5390
Page,0
Campaign,4195
Content,6017
Term,7713


##Vereinheitlichung Sprachniveau

In [ ]:
def get_most_frequent(series):
    mode_series = series.mode(dropna=True)
    if not mode_series.empty:
        return mode_series.iloc[0]
    return None

In [ ]:
for col in df_deals[['City', 'Level of Deutsch']]:
    mode_values = df_deals.groupby('Contact Name')[col].agg(get_most_frequent)
    df_deals[col] = df_deals['Contact Name'].map(mode_values)

In [ ]:
df_deals[['City', 'Level of Deutsch']].isnull().sum()

,0
City,16562
Level of Deutsch,18092


In [ ]:
df_deals['Level of Deutsch'].unique()

array([None, 'б1', 'в1', 'A2', 'в2', 'b1', 'В1', 'B1', 'в1-в2',
       'А2 ( Б1 в июне)', '-', 'B2', 'C2', 'с1', 'Б1', 'Б2',
       'b1 ждет результаты', 'а2', 'Б1( может будет)', 'а1', 'С1',
       'a2 (b1 экзамен 15 июня)', 'а0', 'б2', 'А2',
       'B1 будет в феврале 2025', 'Detmold, Paulinenstraße 95, 32756',
       'Сам оценивает на B2, 13 лет живет в Германии', 'В1-В2',
       'Б1 ( ждет Б2)', 'А2-В1',
       'lэкзамен - 6 июля на В1. курсы вечером (но уверенно говорит на B1)',
       'Гражданка Германии уже год в Германии Учит немецкий и в сентябре b1 через гос-во проходит, а не через ДЖЦ, вечером учится 3 р в неделю с 18 до 21',
       'B1 в процессе обучения',
       'ЯЗ: нем В1 был экз 03.05 повтор и сейчас ждет результаты. Технический англ был. А1 сейчас. ОБР: 2 во информационные и комп сети - инженер системоте',
       'В1 в сентябре', 'Нет', 0, 'Ждем B1', 'А1 сертиф, но по факту А2',
       'a2', 'Пока А2, сдает 17 05 B1', 'окончание 13.06 курса на b1',
       'A1', 'b2', 

In [ ]:
import re

In [ ]:
df_deals['Level of Deutsch'] = df_deals['Level of Deutsch'].astype('string')

In [ ]:
mask = df_deals['Level of Deutsch'].astype(str).str.contains(
    r'\b[A-Z][a-zA-Z]{4,}\b', regex=True, na=False
)

df_deals.loc[mask, ['City', 'Level of Deutsch']] = (
    df_deals.loc[mask, ['Level of Deutsch', 'City']].values)

In [ ]:
trans = str.maketrans(str.maketrans({
    'а': 'A', 'А': 'A',
    'в': 'B', 'В': 'B', 'б': 'B', 'Б': 'B',
    'с': 'C', 'С': 'C',
    'a': 'A', 'b': 'B', 'c': 'C',
    'f': 'A', 'F': 'A'
})
)

trans.update(str.maketrans('abc', 'ABC'))

order = {'A0':0,'A1':1,'A2':2,'B1':3,'B2':4,'C1':5,'C2':6}
pat = re.compile(r'(?i)[abcабс][0-2]?(?:\s*[-–]\s*[abcабс]?[0-2])?\+?')

In [ ]:
def norm(txt):
    if pd.isna(txt): return np.nan
    txt = str(txt).strip().translate(trans).upper().replace(' ', '')
    levels = [m[:2] for m in pat.findall(txt) if len(m) >= 2 and m[:2] in order]
    if not levels:
        return 'A0' if re.search(r'\b0\b', txt, re.I) else np.nan

    return min(levels, key=order.get)
df_deals['Deutsch_Level_Clean'] = df_deals['Level of Deutsch'].apply(norm)

In [ ]:
df_deals[['City', 'Level of Deutsch', 'Deutsch_Level_Clean']].isnull().sum()

,0
City,16562
Level of Deutsch,18092
Deutsch_Level_Clean,18155


In [ ]:
s = df_deals[df_deals['Level of Deutsch'].notna() & df_deals['Deutsch_Level_Clean'].isna()]
s.head(10)

,Id,Deal Owner Name,Closing Date,Quality,Stage,Lost Reason,Page,Campaign,Content,Term,...,Created Time,Course duration,Months of study,Initial Amount Paid,Offer Total Amount,Contact Name,City,Level of Deutsch,SLA Seconds,Deutsch_Level_Clean
328,5805028000055940055,Victor Barnes,NaT,NaN,Registered on Webinar,NaN,/webinar,webinar1906,NaN,invitation\,...,2024-06-17 16:25:00,NaN,NaN,NaN,NaN,5805028000040548184,-,-,NaN,NaN
1751,5805028000052813044,Victor Barnes,NaT,D - Non Target,Call Delayed,NaN,/eng,12.07.2023wide_DE,bloggersvideo14com,wide,...,2024-05-31 10:19:00,6.0,NaN,1000.0,11000.0,5805028000035475298,"Detmold, Paulinenstraße 95, 32756",Detmold,21673.0,NaN
2315,5805028000050843228,Victor Barnes,NaT,D - Non Target,Lost,Stopped Answering,/eng,24.09.23retargeting_DE,v15,retargeting,...,2024-05-20 13:38:00,11.0,NaN,1000.0,11000.0,5805028000050859403,Essen,-,9193.0,NaN
2992,5805028000048965328,Ben Hall,NaT,D - Non Target,Call Delayed,NaN,/eng,12.07.2023wide_DE,bloggersvideo16com,wide,...,2024-05-09 04:37:00,NaN,NaN,NaN,NaN,5805028000047762511,None,Нет,28209.0,NaN
3439,5805028000047733725,Ben Hall,2024-05-08,E - Non Qualified,Lost,Invalid number,/pl-eng,20.03.2024wide_PL,bloggersvideo8com_pl,wide,...,2024-05-03 06:26:00,NaN,NaN,NaN,NaN,5805028000047762511,None,Нет,NaN,NaN
3956,5805028000046850201,Charlie Davis,NaT,C - Low,Waiting For Payment,NaN,/direct,NaN,NaN,NaN,...,2024-04-26 19:06:00,11.0,NaN,1000.0,11000.0,5805028000046830529,Bad Nauheim,-,55390.0,NaN
4851,5805028000044377313,Victor Barnes,NaT,D - Non Target,Lost,Non target,/eng,NaN,NaN,NaN,...,2024-04-16 22:29:00,11.0,NaN,1000.0,11000.0,5805028000044370363,"Thorn-Prikker-Str. 30, Hagen, 58093",Hagen,42227.0,NaN
5574,5805028000043467491,Quincy Vincent,NaT,C - Low,Payment Done,NaN,/direct,NaN,NaN,NaN,...,2024-04-12 13:29:00,11.0,3.0,1000.0,11000.0,5805028000043007272,München,"нулевой уровень, только пошел на курсы.",6383.0,NaN
5779,5805028000043264327,Quincy Vincent,NaT,NaN,Registered on Webinar,NaN,/webinar,webinar1604,NaN,invitation,...,2024-04-11 18:08:00,NaN,NaN,NaN,NaN,5805028000035494714,Kirchlinteln,гражданка,NaN,NaN
5912,5805028000042998237,Rachel White,2024-04-12,E - Non Qualified,Lost,Invalid number,/direct,Jobs_germany_DE,NaN,10_04_2024,...,2024-04-10 20:26:00,NaN,NaN,NaN,NaN,5805028000043007272,München,"нулевой уровень, только пошел на курсы.",NaN,NaN


In [ ]:
s['Level of Deutsch'].unique()

<StringArray>
[                                           '-',
                                      'Detmold',
                                          'Нет',
                                        'Hagen',
      'нулевой уровень, только пошел на курсы.',
                                    'гражданка',
                                    'Paderborn',
                                   '31.05.2024',
                                  'Untersiemau',
                                    'Гражданин',
                      '25 лет живет в Германии',
                      'не сдавал, но гражданин',
                                            '?',
                                            'В',
                                      'не учил',
 'не учила ( разговорный) сразу пошла работать',
                                           '90',
                                           'Бй',
      'разговорный из украины, без сертификата',
                                      'никакой',
      

In [ ]:
df_deals[(df_deals['Level of Deutsch'] == 'Berlin') | (df_deals['Level of Deutsch'] == 'Untersiemau') | (df_deals['Level of Deutsch'] == 'Paderborn')]

,Id,Deal Owner Name,Closing Date,Quality,Stage,Lost Reason,Page,Campaign,Content,Term,...,Created Time,Course duration,Months of study,Initial Amount Paid,Offer Total Amount,Contact Name,City,Level of Deutsch,SLA Seconds,Deutsch_Level_Clean
6915,5805028000040534159,Victor Barnes,NaT,B - Medium,Lost,Changed Decision,/eng/test,NaN,NaN,NaN,...,2024-03-29 20:20:00,11.0,NaN,1000.0,11000.0,5805028000040588059,"Paderborn 33102, Schwabenweg 10",Paderborn,56637.0,NaN
8118,5805028000037284576,Victor Barnes,NaT,D - Non Target,Lost,Next stream,eng/digital-marketing,07.07.23LAL_DE,bloggersvideo9com,LAL1,...,2024-03-13 23:11:00,11.0,NaN,1000.0,11000.0,5805028000037262470,"Lichtenfelser Straße 25, Untersiemau 96253",Untersiemau,40618.0,NaN
18295,5805028000010668160,Oliver Taylor,2023-10-18,A - High,Payment Done,Doesn't Answer,/eng/ux-ui,12.09.23interests_Uxui_DE,bloggersvideo5,interest_work_WebDev,...,2023-10-10 09:52:00,11.0,9.0,1000.0,11000.0,5805028000010678053,B1 немецкий и английский Advance,Berlin,11781.0,NaN


In [ ]:
df_deals.loc[(df_deals['Level of Deutsch'] == 'Berlin'), ['City', 'Level of Deutsch']] = (
    df_deals.loc[(df_deals['Level of Deutsch'] == 'Berlin'), ['Level of Deutsch', 'City']].values)

In [ ]:
df_deals[(df_deals['Level of Deutsch'] == 'Berlin')]

,Id,Deal Owner Name,Closing Date,Quality,Stage,Lost Reason,Page,Campaign,Content,Term,...,Created Time,Course duration,Months of study,Initial Amount Paid,Offer Total Amount,Contact Name,City,Level of Deutsch,SLA Seconds,Deutsch_Level_Clean


In [ ]:
df_deals['Deutsch_Level_Clean'].unique()

array([nan, 'B1', 'A2', 'B2', 'C2', 'C1', 'A1', 'A0'], dtype=object)

In [ ]:
df_deals.drop(columns=['Level of Deutsch'], inplace=True)

In [ ]:
df_deals.isna().sum()

,0
Id,0
Deal Owner Name,29
Closing Date,6600
Quality,2231
Stage,0
Lost Reason,5390
Page,0
Campaign,4195
Content,6017
Term,7713


In [ ]:
cols = ['Deal Owner Name', 'Quality', 'Lost Reason', 'Campaign',
        'Content', 'Term', 'Source', 'Payment Type', 'Product',
        'Education Type', 'City', 'Deutsch_Level_Clean' ]
df_deals[cols] = df_deals[cols].fillna('Unknown')

In [ ]:
df_deals.head()

,Id,Deal Owner Name,Closing Date,Quality,Stage,Lost Reason,Page,Campaign,Content,Term,...,Education Type,Created Time,Course duration,Months of study,Initial Amount Paid,Offer Total Amount,Contact Name,City,SLA Seconds,Deutsch_Level_Clean
0,5805028000056864695,Ben Hall,NaT,Unknown,New Lead,Unknown,/eng/test,03.07.23women,v16,women,...,Unknown,2024-06-21 15:30:00,NaN,NaN,NaN,NaN,5805028000056849495,Unknown,NaN,Unknown
1,5805028000056859489,Ulysses Adams,NaT,Unknown,New Lead,Unknown,/at-eng,Unknown,Unknown,Unknown,...,Morning,2024-06-21 15:23:00,6.0,NaN,NaN,2000.0,5805028000056834471,Unknown,NaN,Unknown
2,5805028000056832357,Ulysses Adams,2024-06-21 14:45:00,D - Non Target,Lost,Non target,/at-eng,engwien_AT,b1-at,21_06_2024,...,Unknown,2024-06-21 00:00:00,NaN,NaN,NaN,NaN,5805028000056854421,Unknown,1603.0,Unknown
3,5805028000056824246,Eva Kent,2024-06-21 13:32:00,E - Non Qualified,Lost,Invalid number,/eng,04.07.23recentlymoved_DE,bloggersvideo14com,recentlymoved,...,Unknown,2024-06-21 00:00:00,NaN,NaN,NaN,NaN,5805028000056889351,Unknown,3604.0,Unknown
4,5805028000056873292,Ben Hall,2024-06-21 13:21:00,D - Non Target,Lost,Non target,/eng,discovery_DE,website,Unknown,...,Unknown,2024-06-21 00:00:00,NaN,NaN,NaN,NaN,5805028000056876176,Unknown,3192.0,Unknown


In [ ]:
missing_percent = df_deals.isnull().mean() * 100
missing_percent = missing_percent.round(2)

subset = df_deals.isnull().head(300).astype(int)

fig = go.Figure(data=go.Heatmap(
    z=subset.values,
    x=[f"{col}<br>({missing_percent[col]}%)" for col in df_deals.columns],  # подписи с %
    colorscale=[[0, 'white'], [1, 'black']],
    showscale=False
))

fig.update_layout(
    title="Hitmap of missing values",
    height=500,
)

fig.show()

In [ ]:
df_deals.info()

<class 'pandas.core.frame.DataFrame'>
Index: 19657 entries, 0 to 21592
Data columns (total 23 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Id                   19657 non-null  object        
 1   Deal Owner Name      19657 non-null  object        
 2   Closing Date         13057 non-null  datetime64[ns]
 3   Quality              19657 non-null  object        
 4   Stage                19657 non-null  object        
 5   Lost Reason          19657 non-null  object        
 6   Page                 19657 non-null  object        
 7   Campaign             19657 non-null  object        
 8   Content              19657 non-null  object        
 9   Term                 19657 non-null  object        
 10  Source               19657 non-null  object        
 11  Payment Type         19657 non-null  object        
 12  Product              19657 non-null  object        
 13  Education Type       19657 non-null 

#4.2 Deals
##Deskriptive Statistik:
1. Berechnen Sie zusammenfassende Statistiken (Mittelwert, Median, Modus,
Spannweite) für numerische Felder.
2. Analysieren Sie kategoriale Felder wie Qualität, Phase, Quelle und Produkt.

##Deskriptive Statistik für Numerische Feleder

In [ ]:
df_deals.describe().round(2)

,Closing Date,Created Time,Course duration,Months of study,Initial Amount Paid,Offer Total Amount,SLA Seconds
count,13057,19657,3513.00,835.00,3220.00,3267.00,14857.00
mean,2024-01-28 11:11:48.410814208,2024-01-27 18:49:19.205372160,10.20,5.45,1168.98,8964.71,111722.39
min,2023-07-03 17:03:00,2022-10-11 00:00:00,6.00,0.00,9.00,1200.00,3.00
25%,2023-11-11 00:00:00,2023-11-18 14:47:00,11.00,3.00,1000.00,5000.00,4358.00
50%,2024-02-08 00:00:00,2024-02-05 20:37:00,11.00,5.00,1000.00,11000.00,19603.00
75%,2024-04-18 00:00:00,2024-04-14 10:22:00,11.00,8.00,1000.00,11000.00,55752.00
max,2024-12-11 00:00:00,2024-06-21 15:30:00,11.00,11.00,11000.00,11500.00,26908464.00
std,NaN,NaN,1.84,2.92,1387.01,3251.08,729220.04


##Deskriptive Statistik für Categorische Feleder

In [ ]:
categ_cols = ['Deal Owner Name','Quality', 'Stage', 'Lost Reason', 'Source', 'Payment Type', 'Product', 'Education Type']

In [ ]:
descr_stat(df_deals, cols=categ_cols)



=== Deal Owner Name (unique: 28, total: 19657) ===
          value  count  percent
  Charlie Davis   2788    14.18
   Julia Nelson   2061    10.48
  Ulysses Adams   2057    10.46
 Quincy Vincent   1782     9.07
Paula Underwood   1749     8.90
       Ben Hall   1287     6.55
     Nina Scott   1213     6.17
  Victor Barnes   1182     6.01
   Cara Iverson   1002     5.10
    Diana Evans    923     4.70
     Jane Smith    904     4.60
   Kevin Parker    500     2.54
       Eva Kent    445     2.26
     Ian Miller    443     2.25
   Rachel White    341     1.73
  Mason Roberts    258     1.31
      Bob Brown    230     1.17
  Oliver Taylor    160     0.81
    George King     84     0.43
   Yara Edwards     73     0.37
      Sam Young     55     0.28
      Amy Green     49     0.25
        Unknown     29     0.15
  Alice Johnson     25     0.13
       John Doe     12     0.06
    Xander Dean      3     0.02
    Wendy Clark      1     0.01
 Zachary Foster      1     0.01


=== Quality (uniq

##Diagramme

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fields = ['Deal Owner Name', 'Quality', 'Source', 'Stage', 'Lost Reason', 'Product', 'Payment Type',  'Education Type']

fig = make_subplots(
    rows=2, cols=4,
    subplot_titles=fields,
    vertical_spacing=0.10,
    horizontal_spacing=0.08
)

for i, field in enumerate(fields):
    row = i // 4 + 1
    col = i % 4 + 1

    counts = df_deals[field].value_counts().head(28)

    fig.add_trace(
        go.Bar(
            x=counts.values,
            y=counts.index,
            orientation='h',
            name=field,
            marker_color='plum',
            text=counts.values,
            textposition='auto'
        ),
        row=row, col=col
    )

fig.update_layout(
    title_text="Распределение по полям",
    height=950,
    showlegend=False,
    plot_bgcolor='white'
)

fig.show()

##Datenspeicherung

In [ ]:
df_deals.to_pickle('Cl_Deals.pkl')